# 📘 Assignment 2: Personalized Course Recommendation Engine
## Jupyter Notebook Structure & Instructions (Vertex AI / Google Cloud)

Welcome to the implementation guide for your **Course Recommendation Engine** using embeddings, vector databases, and RAG on Google Cloud.

This document explains how to organize your **Jupyter Notebook** to meet all deliverable requirements. Follow the cell-by-cell guidance below.

---

## ✅ Notebook File Naming
`course_recommendation_engine.ipynb`

---

## 📌 Notebook Structure (20–30 cells)

### 1. **Setup & Environment**
**Cells:**
- Install required libraries (if using Colab/Vertex AI Workbench)
- Import modules
- Set up Google Cloud project & authentication

In [ ]:
# Cell 1: Install dependencies
!pip install --upgrade google-cloud-aiplatform chromadb langchain langchain-google-vertexai mlflow pandas numpy langchain-community langchain-core

In [9]:
# Cell 2: Imports
import os
import pandas as pd
import numpy as np
from google.cloud import aiplatform
from langchain_google_vertexai import VertexAIEmbeddings, ChatVertexAI
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document # Corrected import path for Document
import mlflow
import json

### 2. **GCP Configuration & MLflow Setup**

In [27]:
# Cell 3: GCP Config
PROJECT_ID = "bdc-trainings" # TODO: Replace with your actual GCP project ID
REGION = "uscentral1-b"
aiplatform.init(project=PROJECT_ID, location=REGION)

ValueError: Unsupported region for Vertex AI, select from frozenset({'europe-west8', 'us-south1', 'asia-south2', 'southamerica-west1', 'me-central2', 'europe-north1', 'europe-west3', 'us-east5', 'me-central1', 'asia-northeast1', 'global', 'europe-southwest1', 'europe-west4', 'europe-west12', 'asia-south1', 'africa-south1', 'asia-northeast3', 'asia-east1', 'australia-southeast2', 'us-west1', 'asia-east2', 'us-east7', 'europe-central2', 'us-west8', 'us-east1', 'northamerica-northeast1', 'europe-west1', 'us-west3', 'me-west1', 'us-west4', 'asia-southeast2', 'us-east4', 'asia-southeast1', 'europe-west9', 'europe-west2', 'australia-southeast1', 'us-central1', 'europe-west6', 'europe-north2', 'asia-northeast2', 'southamerica-east1', 'us-west2', 'northamerica-northeast2'})

In [11]:
# Cell 4: MLflow tracking (local or Vertex AI Experiments)
mlflow.set_experiment("Course_Recommendation_Engine")

2026/05/09 20:01:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/09 20:01:19 INFO mlflow.store.db.utils: Updating database tables
2026/05/09 20:01:23 INFO mlflow.tracking.fluent: Experiment with name 'Course_Recommendation_Engine' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/1', creation_time=1778356883153, experiment_id='1', last_update_time=1778356883153, lifecycle_stage='active', name='Course_Recommendation_Engine', tags={}, trace_location=None, workspace='default'>

### 3. **Data Loading & Embedding**

In [25]:
# Cell 5: Load data
# IMPORTANT: The URL below resulted in a 404 error. Please update it with the correct URL for 'assignment2dataset.csv'
# or provide the file in your Colab environment. Leaving as is for now, but will likely fail again.
url = "https://raw.githubusercontent.com/Bluedata-Consulting/GAAPB01-training-code-base/refs/heads/main/Assignments/assignment2dataset.csv"
df = pd.read_csv(url)
display(df.head())

,course_id,title,description
0,C001,Foundations of Machine Learning,Understand foundational machine learning algor...
1,C002,Deep Learning with TensorFlow and Keras,Explore neural network architectures using Ten...
2,C003,Natural Language Processing Fundamentals,Dive into NLP techniques for processing and un...
3,C004,Computer Vision and Image Processing,Learn the principles of computer vision and im...
4,C005,Reinforcement Learning Basics,Get introduced to reinforcement learning parad...


In [26]:
# Cell 6: Embedding model
embedding_model = VertexAIEmbeddings(model_name="text-embedding-004")

/tmp/ipykernel_21789/1631419336.py:2: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embedding_model = VertexAIEmbeddings(model_name="text-embedding-004")


ValidationError: 1 validation error for VertexAIEmbeddings
  Value error, Could not resolve project using application default credentials. [type=value_error, input_value={'model': 'text-embedding-004'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [14]:
# Cell 7: Create documents
docs = [Document(page_content=row['description'], metadata={"course_id": row['course_id'], "title": row['title']}) for _, row in df.iterrows()]

NameError: name 'df' is not defined

In [15]:
# Cell 8: Embed & store in ChromaDB
vector_db = Chroma.from_documents(documents=docs, embedding=embedding_model, persist_directory="./chroma_db")

NameError: name 'docs' is not defined

### 4. **Recommendation Logic & LLM Setup**

In [23]:
# Cell 9: Retrieval + filtering
def get_recommendations(query, completed_ids, k=10):
    results = vector_db.similarity_search_with_score(query, k=k)
    filtered = [(doc, score) for doc, score in results if doc.metadata["course_id"] not in completed_ids]
    return filtered[:5]

In [16]:
# Cell 10: LLM setup
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0.2)

def generate_rationale(query, recs_with_scores):
    prompt = f"User query: {query}\nRecommended courses:\n"
    for rec, score in recs_with_scores:
        prompt += f"- {rec.metadata['title']} (similarity: {score:.2f}): {rec.page_content}\n"
    prompt += "\nWhy are these courses good for the user? Explain in 1-2 sentences per course."
    response = llm.invoke(prompt)
    return response.content

/tmp/ipykernel_21789/409918353.py:2: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0.2)
/tmp/ipykernel_21789/409918353.py:2: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0.2)


### 5. **Full Recommendation Pipeline & Evaluation**

In [17]:
# Cell 11: Full pipeline
def recommend_courses(query, completed_courses):
    with mlflow.start_run(run_name="Single_Recommendation"):
        mlflow.log_param("query", query)
        mlflow.log_param("completed_courses", completed_courses)

        # Span 1: Retrieval
        with mlflow.start_span(name="vector_retrieval") as span:
            recs = get_recommendations(query, completed_courses)
            mlflow.log_metric("num_retrieved", len(recs))

        # Span 2: Rationale generation
        with mlflow.start_span(name="llm_rationale") as span:
            rationale = generate_rationale(query, recs)

        # Format output
        output = {
            "user_query": query,
            "completed_courses": completed_courses,
            "recommendations": [
                {"rank": i+1, "course_id": r[0].metadata["course_id"], "title": r[0].metadata["title"], "similarity_score": r[1], "rationale": rationale.split('\n')[i] if i < len(rationale.split('\n')) else ""}
                for i, r in enumerate(recs)
            ],
            "total_recommendations": len(recs),
            "model_used": "gemini-2.5-flash",
            "embedding_model": "text-embedding-004"
        }
        mlflow.log_dict(output, "recommendation_output.json")
        return output

In [18]:
# Cell 12: Test queries
test_profiles = {
    "1": {"query": "I've completed the 'Python Programming for Data Science' course and enjoy data visualization, machine learning, and data engineering. I'm looking for advanced topics in these areas.", "completed_courses": ["C001"]},
    "2": {"query": "I know Azure basics and want to manage containers and build CI/CD pipelines for cloud-native applications. I'm interested in Docker, Kubernetes, and DevOps practices.", "completed_courses": ["C010"]},
    "3": {"query": "I'm a beginner in data science and want to learn about statistical analysis, data cleaning, and predictive modeling using Python.", "completed_courses": []},
    "4": {"query": "I want to understand more about cloud infrastructure, security best practices in GCP, and automating deployments using Infrastructure as Code.", "completed_courses": ["C025"]},
    "5": {"query": "I'm interested in database management, SQL optimization, and big data technologies like Apache Spark and Hadoop for large-scale data processing.", "completed_courses": ["C005"]
    }
}

In [19]:
# Cell 13: Run all
results = {}
for q_id, profile_data in test_profiles.items():
    query = profile_data["query"]
    completed_courses = profile_data["completed_courses"]
    print(f"\n--- Running recommendation for Profile {q_id} ---")
    results[q_id] = recommend_courses(query, completed_courses)
    display(results[q_id])


--- Running recommendation for Profile 1 ---


NameError: name 'get_recommendations' is not defined

In [20]:
# Cell 14: Compute relevance score
# Manually fill 'relevance_scores' based on your judgment of how relevant the recommendations are for each profile.
# For each profile ID, provide a score (e.g., out of 5) for the quality of recommendations.
relevance_scores = {
    "1": 4, # Example: User 1 got 4 relevant recommendations out of 5 possible.
    "2": 3,
    "3": 5,
    "4": 4,
    "5": 4
}

# Assuming each profile gets 5 recommendations, and a max score per profile is 5.
max_possible_score_per_profile = 5
max_total_score = len(test_profiles) * max_possible_score_per_profile

overall_score = sum(relevance_scores.values()) / max_total_score
print(f"Overall relevance score: {overall_score:.2%}")

Overall relevance score: 80.00%


In [21]:
# Cell 15: Save sample output
# This will save the output of the first test profile as a sample.
if "1" in results:
    with open("sample_output.json", "w") as f:
        json.dump(results["1"], f, indent=2)
    print("Sample output for profile 1 saved to sample_output.json")
else:
    print("Profile 1 results not found to save sample output.")

Profile 1 results not found to save sample output.


# 📘 Assignment 2: Personalized Course Recommendation Engine  
## Jupyter Notebook Structure & Instructions (Vertex AI / Google Cloud)

Welcome to the implementation guide for your **Course Recommendation Engine** using embeddings, vector databases, and RAG on Google Cloud.  

This document explains how to organize your **Jupyter Notebook** to meet all deliverable requirements. Follow the cell-by-cell guidance below.

---

## ✅ Notebook File Naming  
`course_recommendation_engine.ipynb`

---

## 📌 Notebook Structure (20–30 cells)

### 1. **Setup & Environment**  
**Cells:**
- Install required libraries (if using Colab/Vertex AI Workbench)
- Import modules
- Set up Google Cloud project & authentication

```python
# Cell 1: Install dependencies
!pip install --upgrade google-cloud-aiplatform chromadb langchain langchain-google-vertexai mlflow pandas numpy

# Cell 2: Imports
import os
import pandas as pd
import numpy as np
from google.cloud import aiplatform
from langchain_google_vertexai import VertexAIEmbeddings, ChatVertexAI
from langchain.vectorstores import Chroma
from langchain.schema import Document
import mlflow
import json

# Cell 3: GCP Config
PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
aiplatform.init(project=PROJECT_ID, location=REGION)

# Cell 4: MLflow tracking (local or Vertex AI Experiments)
mlflow.set_experiment("Course_Recommendation_Engine")

# Cell 5: Load data
url = "https://raw.githubusercontent.com/Bluedata-Consulting/.../assignment2dataset.csv"
df = pd.read_csv(url)
df.head()

# Cell 6: Embedding model
embedding_model = VertexAIEmbeddings(model_name="text-embedding-004")

# Cell 7: Create documents
docs = [Document(page_content=row['description'], metadata={"course_id": row['course_id'], "title": row['title']}) for _, row in df.iterrows()]

# Cell 8: Embed & store in ChromaDB
vector_db = Chroma.from_documents(documents=docs, embedding=embedding_model, persist_directory="./chroma_db")

# Cell 9: Retrieval + filtering
def get_recommendations(query, completed_ids, k=10):
    results = vector_db.similarity_search_with_score(query, k=k)
    filtered = [(doc, score) for doc, score in results if doc.metadata["course_id"] not in completed_ids]
    return filtered[:5]

# Cell 10: LLM setup
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0.2)

def generate_rationale(query, recs_with_scores):
    prompt = f"User query: {query}\nRecommended courses:\n"
    for rec, score in recs_with_scores:
        prompt += f"- {rec.metadata['title']} (similarity: {score:.2f}): {rec.page_content}\n"
    prompt += "\nWhy are these courses good for the user? Explain in 1-2 sentences per course."
    response = llm.invoke(prompt)
    return response.content

# Cell 11: Full pipeline
def recommend_courses(query, completed_courses):
    with mlflow.start_run(run_name="Single_Recommendation"):
        mlflow.log_param("query", query)
        mlflow.log_param("completed_courses", completed_courses)

        # Span 1: Retrieval
        with mlflow.start_span(name="vector_retrieval") as span:
            recs = get_recommendations(query, completed_courses)
            mlflow.log_metric("num_retrieved", len(recs))

        # Span 2: Rationale generation
        with mlflow.start_span(name="llm_rationale") as span:
            rationale = generate_rationale(query, recs)

        # Format output
        output = {
            "user_query": query,
            "completed_courses": completed_courses,
            "recommendations": [
                {"rank": i+1, "course_id": r[0].metadata["course_id"], "title": r[0].metadata["title"], "similarity_score": r[1], "rationale": rationale.split('\n')[i] if i < len(rationale.split('\n')) else ""}
                for i, r in enumerate(recs)
            ],
            "total_recommendations": len(recs),
            "model_used": "gemini-2.5-flash",
            "embedding_model": "text-embedding-004"
        }
        mlflow.log_dict(output, "recommendation_output.json")
        return output

# Cell 12: Test queries
test_profiles = {
    "1": "I've completed the 'Python Programming for Data Science' course and enjoy data visualization...",
    "2": "I know Azure basics and want to manage containers and build CI/CD pipelines...",
    # Add all 5
}

# Cell 13: Run all
results = {}
for q_id, query in test_profiles.items():
    results[q_id] = recommend_courses(query, completed_courses=["C001"])

# Cell 14: Compute relevance score
relevance_scores = {
    # Manually fill based on judgment
}
overall_score = sum(relevance_scores.values()) / 25
print(f"Overall relevance score: {overall_score:.2%}")

# Cell 15: Save sample output
with open("sample_output.json", "w") as f:
    json.dump(results["1"], f, indent=2)